<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/wav2vec2_speechbrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

# # 작업 경로 설정
# #%cd /content/drive/MyDrive/코딩공부/project_folder
# #!git clone https://github.com/speechbrain/speechbrain.git
# %cd /content/drive/MyDrive/코딩공부/project_folder/speechbrain/recipes/LibriSpeech/ASR/CTC/


In [3]:
# !pip install -r extra_requirements.txt

In [4]:
# !pip install hyperpyyaml
# !pip install speechbrain
# !pip install torchaudio

In [5]:
# # 다른 조건 사용시 yaml 수정 필요

# !python train_with_wav2vec.py hparams/downsampled/train_hf_wavlm_signal_downsampling.yaml --downsampling_factor 2

In [6]:
# import os
# from pathlib import Path

# def collect_wav_files(audio_root, audio_ext="flac"):
#     """오디오 파일 경로와 ID 수집"""
#     wav_files = {}
#     for path in Path(audio_root).rglob(f"*.{audio_ext}"):
#         utt_id = path.stem
#         wav_files[utt_id] = path
#     return wav_files

# def collect_trans_ids(audio_root):
#     """전사 파일에서 ID 수집"""
#     trans_ids = set()
#     for trans_file in Path(audio_root).rglob("*.trans.txt"):
#         with open(trans_file, 'r', encoding='utf-8') as f:
#             for line in f:
#                 utt_id = line.strip().split()[0]
#                 trans_ids.add(utt_id)
#     return trans_ids

# def remove_missing_transcriptions(audio_root, audio_ext="flac"):
#     """전사 정보가 없는 오디오를 삭제"""
#     wav_files = collect_wav_files(audio_root, audio_ext)
#     trans_ids = collect_trans_ids(audio_root)

#     missing = [utt_id for utt_id in wav_files if utt_id not in trans_ids]

#     if missing:
#         print(f"⚠️ 전사 누락 파일 {len(missing)}개를 삭제합니다...")
#         for utt_id in sorted(missing):
#             path = wav_files[utt_id]
#             try:
#                 os.remove(path)
#                 print(f"🗑️ 삭제됨: {path}")
#             except Exception as e:
#                 print(f"❌ 삭제 실패: {path} ({e})")
#     else:
#         print("✅ 모든 오디오에 전사 정보가 있습니다.")

# # ✅ 사용 예시
# audio_root = "/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech"  # 변경 필요
# remove_missing_transcriptions(audio_root, audio_ext="flac")


In [7]:
# import pandas as pd

# # 중복 제거할 CSV 파일 경로 지정
# csv_path = "/content/drive/MyDrive/코딩공부/project_folder/results/wav2vec_train_result/results/train_wav2vec2_char/<seed>/train.csv"  # 필요 시 valid.csv, test.csv로 변경

# # CSV 불러오기
# df = pd.read_csv(csv_path)

# # ID 기준 중복 제거 (첫 번째 것만 유지)
# df_dedup = df.drop_duplicates(subset="ID", keep="first")

# # 결과 확인 (선택)
# print(f"Before: {len(df)} rows, After: {len(df_dedup)} rows")

# # 덮어쓰기 저장 (백업하고 싶으면 따로 저장 가능)
# df_dedup.to_csv(csv_path, index=False)

# 인퍼런스

In [8]:
!pip install torchaudio transformers jiwer

In [9]:
import os
import torchaudio
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from jiwer import wer

# ✅ 1. 모델 설정
model_name = "facebook/wav2vec2-base-960h"  # Transformers에서 지원되는 모델
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)
model.eval()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [10]:
# import os
# import torchaudio
# import torch
# from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
# from jiwer import wer

# # ✅ 1. 모델 로딩 (영어 LibriSpeech에 최적화된 모델)
# model_name = "facebook/wav2vec2-base-960h"  # 또는 large-960h-lv60
# processor = Wav2Vec2Processor.from_pretrained(model_name)
# model = Wav2Vec2ForCTC.from_pretrained(model_name)
# model.eval()

# # ✅ 2. 테스트셋 경로 설정
# base_dir = "/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/test-clean"

# # ✅ 3. 오디오 로딩 함수
# def load_audio(path):
#     waveform, sr = torchaudio.load(path)
#     if sr != 16000:
#         resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
#         waveform = resampler(waveform)
#     return waveform.squeeze()

# # ✅ 4. 인퍼런스 함수
# def transcribe(wav_tensor):
#     inputs = processor(wav_tensor, sampling_rate=16000, return_tensors="pt", padding=True)
#     with torch.no_grad():
#         logits = model(**inputs).logits
#     pred_ids = torch.argmax(logits, dim=-1)
#     transcription = processor.batch_decode(pred_ids)[0]
#     return transcription.lower()

# # ✅ 5. 평가 루프
# total_wer = 0
# total_cnt = 0
# missing_transcripts = []

# for book_id in os.listdir(base_dir):
#     book_path = os.path.join(base_dir, book_id)
#     if not os.path.isdir(book_path):
#         continue

#     for chapter_id in os.listdir(book_path):
#         chapter_path = os.path.join(book_path, chapter_id)
#         if not os.path.isdir(chapter_path):
#             continue

#         # transcript 파일 이름은 "{book_id}-{chapter_id}.trans.txt"
#         transcript_file = f"{book_id}-{chapter_id}.trans.txt"
#         transcript_path = os.path.join(chapter_path, transcript_file)

#         if not os.path.exists(transcript_path):
#             print(f"🚫 transcript 파일 없음: {transcript_path}")
#             missing_transcripts.append(transcript_path)
#             continue

#         with open(transcript_path, 'r', encoding='utf-8') as f:
#             lines = [line.strip() for line in f if line.strip()]

#         for line in lines:
#             parts = line.split(maxsplit=1)
#             if len(parts) != 2:
#                 continue
#             wav_base, gt_text = parts
#             wav_path = os.path.join(chapter_path, wav_base + ".flac")

#             if not os.path.exists(wav_path):
#                 print(f"⚠️ 음성 파일 없음: {wav_path}")
#                 continue

#             # 오디오 로딩 및 인퍼런스
#             wav_tensor = load_audio(wav_path)
#             pred_text = transcribe(wav_tensor)

#             # WER 계산
#             sample_wer = wer(gt_text.lower(), pred_text)
#             total_wer += sample_wer
#             total_cnt += 1

#             print(f"\n🎧 {wav_base}.flac")
#             print(f"GT   : {gt_text}")
#             print(f"PRED : {pred_text}")
#             print(f"WER  : {sample_wer:.3f}")

# # ✅ 6. 최종 결과 출력
# print("\n======================")
# if total_cnt > 0:
#     avg_wer = total_wer / total_cnt
#     print(f"📊 전체 평균 WER: {avg_wer:.3f}")
# else:
#     print("⚠️ 평가할 샘플이 없습니다.")

# if missing_transcripts:
#     print("\n❗ 누락된 transcript 파일 목록:")
#     for path in missing_transcripts:
#         print(" -", path)


In [11]:
# import os
# import torchaudio
# import torch
# from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
# from jiwer import wer

# # ✅ 1. 모델 로딩 (XLSR 버전 - 다국어 pretrain 모델)
# model_name = "jonatasgrosman/wav2vec2-large-xlsr-53-english"  # or fine-tuned version for your language
# processor = Wav2Vec2Processor.from_pretrained(model_name)
# model = Wav2Vec2ForCTC.from_pretrained(model_name)
# model.eval()

# # ✅ 2. 테스트셋 경로 설정
# base_dir = "/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/test-clean"

# # ✅ 3. 오디오 로딩 함수
# def load_audio(path):
#     waveform, sr = torchaudio.load(path)
#     if sr != 16000:
#         resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
#         waveform = resampler(waveform)
#     return waveform.squeeze()

# # ✅ 4. 인퍼런스 함수
# def transcribe(wav_tensor):
#     inputs = processor(wav_tensor, sampling_rate=16000, return_tensors="pt", padding=True)
#     with torch.no_grad():
#         logits = model(**inputs).logits
#     pred_ids = torch.argmax(logits, dim=-1)
#     transcription = processor.batch_decode(pred_ids)[0]
#     return transcription.lower()

# # ✅ 5. 평가 루프
# total_wer = 0
# total_cnt = 0
# missing_transcripts = []

# for book_id in os.listdir(base_dir):
#     book_path = os.path.join(base_dir, book_id)
#     if not os.path.isdir(book_path):
#         continue

#     for chapter_id in os.listdir(book_path):
#         chapter_path = os.path.join(book_path, chapter_id)
#         if not os.path.isdir(chapter_path):
#             continue

#         transcript_file = f"{book_id}-{chapter_id}.trans.txt"
#         transcript_path = os.path.join(chapter_path, transcript_file)

#         if not os.path.exists(transcript_path):
#             print(f"🚫 transcript 파일 없음: {transcript_path}")
#             missing_transcripts.append(transcript_path)
#             continue

#         with open(transcript_path, 'r', encoding='utf-8') as f:
#             lines = [line.strip() for line in f if line.strip()]

#         for line in lines:
#             parts = line.split(maxsplit=1)
#             if len(parts) != 2:
#                 continue
#             wav_base, gt_text = parts
#             wav_path = os.path.join(chapter_path, wav_base + ".flac")

#             if not os.path.exists(wav_path):
#                 print(f"⚠️ 음성 파일 없음: {wav_path}")
#                 continue

#             wav_tensor = load_audio(wav_path)
#             pred_text = transcribe(wav_tensor)

#             sample_wer = wer(gt_text.lower(), pred_text)
#             total_wer += sample_wer
#             total_cnt += 1

#             print(f"\n🎧 {wav_base}.flac")
#             print(f"GT   : {gt_text}")
#             print(f"PRED : {pred_text}")
#             print(f"WER  : {sample_wer:.3f}")

# # ✅ 6. 결과 출력
# print("\n======================")
# if total_cnt > 0:
#     avg_wer = total_wer / total_cnt
#     print(f"📊 전체 평균 WER: {avg_wer:.3f}")
# else:
#     print("⚠️ 평가할 샘플이 없습니다.")

# if missing_transcripts:
#     print("\n❗ 누락된 transcript 파일 목록:")
#     for path in missing_transcripts:
#         print(" -", path)


#파인튜닝

In [12]:
import os
import csv

# base_dir = "/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/train-clean-100"  # 여기에 본인의 경로 입력
# output_csv = "train_clean_100.csv"

# with open(output_csv, "w", newline='', encoding="utf-8") as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow(["path", "text"])  # 헤더

#     for book_id in os.listdir(base_dir):
#         book_path = os.path.join(base_dir, book_id)
#         if not os.path.isdir(book_path):
#             continue

#         for chapter_id in os.listdir(book_path):
#             chapter_path = os.path.join(book_path, chapter_id)
#             if not os.path.isdir(chapter_path):
#                 continue

#             transcript_file = os.path.join(chapter_path, f"{book_id}-{chapter_id}.trans.txt")
#             if not os.path.exists(transcript_file):
#                 continue

#             with open(transcript_file, "r", encoding="utf-8") as f:
#                 for line in f:
#                     parts = line.strip().split(maxsplit=1)
#                     if len(parts) != 2:
#                         continue
#                     wav_id, text = parts
#                     wav_path = os.path.join(chapter_path, wav_id + ".flac")
#                     writer.writerow([wav_path, text])


In [13]:
from datasets import load_dataset, Audio
from datasets import Dataset
import pandas as pd
# df = pd.read_csv("/content/train_clean_100.csv")
# dataset = Dataset.from_pandas(df)
# dataset = dataset.cast_column("path", Audio(sampling_rate=16000))

In [14]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

model_name = "jonatasgrosman/wav2vec2-large-xlsr-53-english"

processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

In [15]:
# # def prepare_dataset(batch):
# #     audio = batch["path"]
# #     inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_attention_mask=True)
# #     batch["input_values"] = inputs.input_values[0]
# #     batch["attention_mask"] = inputs.attention_mask[0]

# #     with processor.as_target_processor():
# #         batch["labels"] = processor(batch["text"]).input_ids
# #     return batch
# # def prepare_dataset(batch):
# #     audio = batch["path"]
# #     inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_attention_mask=True)
# #     batch["input_values"] = inputs.input_values[0]
# #     with processor.as_target_processor():
# #         batch["labels"] = processor(batch["text"]).input_ids
# #     return batch

# # dataset = dataset.map(
# #     prepare_dataset,
# #     remove_columns=dataset.column_names,
# #     batched=False,      # 한 샘플씩 처리 → 메모리 안전
# #     num_proc=1          # CPU 1개만 사용 → 메모리 폭발 방지
# # )

# # dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names, num_proc=1)
# # dataset.save_to_disk("/content/drive/MyDrive/코딩공부/project_folder/results/wav2vec_train_result/pretrained")

# # ✅ 전처리 함수 정의
# def prepare_dataset(batch):
#     audio = batch["path"]  # path는 datasets에서 {"array": ..., "sampling_rate": ...} 포함된 dict라고 가정
#     inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_attention_mask=True)
#     batch["input_values"] = inputs.input_values[0]

#     with processor.as_target_processor():
#         batch["labels"] = processor(batch["text"]).input_ids
#     return batch

# # ✅ chunk 단위로 처리할 설정
# chunk_size = 1000
# save_dir = "/content/drive/MyDrive/코딩공부/processed_chunks"
# os.makedirs(save_dir, exist_ok=True)

# n = len(dataset)
# processed_chunk_paths = []

# # ✅ chunk별로 처리, 저장
# for i in range(0, n, chunk_size):
#     print(f"🔄 Processing chunk {i} ~ {min(i + chunk_size, n)}...")

#     # 1. 조각 선택
#     chunk = dataset.select(range(i, min(i + chunk_size, n)))

#     # 2. 전처리
#     chunk = chunk.map(prepare_dataset, batched=False, remove_columns=dataset.column_names)

#     # 3. 저장
#     chunk_path = os.path.join(save_dir, f"chunk_{i // chunk_size}")
#     chunk.save_to_disk(chunk_path)
#     processed_chunk_paths.append(chunk_path)

# print("✅ 모든 조각 처리 및 저장 완료!")

In [16]:
# from datasets import load_from_disk, concatenate_datasets
# # ✅ 병합
# print("📦 저장된 조각 병합 중...")
# chunks = [load_from_disk(path) for path in processed_chunk_paths]
# final_dataset = concatenate_datasets(chunks)

# # ✅ 최종 저장
# final_save_path = "/content/drive/MyDrive/코딩공부/project_folder/results/wav2vec_train_result/preprocessed"
# final_dataset.save_to_disk(final_save_path)

# print(f"🎉 최종 데이터셋 저장 완료: {final_save_path}")

In [17]:
!pip install --upgrade datasets numpy pyarrow

In [18]:
from datasets import load_from_disk

# 경로: 병합된 데이터셋을 save_to_disk() 했던 폴더
load_path = "/content/drive/MyDrive/코딩공부/project_folder/results/wav2vec_train_result/preprocessed"


# 병합된 최종 데이터셋 로드
final_dataset = load_from_disk(load_path)

# 포맷 반드시 지정
final_dataset.set_format(type="torch", columns=["input_values", "labels"])  # ← ✅ 이거 꼭 필요!



Loading dataset from disk:   0%|          | 0/44 [00:00<?, ?it/s]

In [19]:
# 확인
print(final_dataset)
print(final_dataset[0])

Dataset({
    features: ['input_values', 'labels'],
    num_rows: 26990
})
{'input_values': tensor([1.0157e-03, 1.0157e-03, 1.4864e-03,  ..., 7.4445e-05, 7.4445e-05,
        5.4509e-04]), 'labels': tensor([3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3,
        4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3,
        3, 4, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 5, 3, 4, 3, 3, 3, 3, 4, 3, 3,
        3, 4, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 4, 3, 3, 3,
        3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 4,
        3, 4, 3, 4, 3, 4, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3])}


In [20]:
from dataclasses import dataclass
from transformers import DataCollatorWithPadding
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: str = "longest"

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)


In [21]:
import numpy as np
from jiwer import wer

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    return {"wer": wer(label_str, pred_str)}


In [22]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/코딩공부/project_folder/results/wav2vec_train_result/results/train_wav2vec2_char/1986/pretrained_model",
    group_by_length=False,
    per_device_train_batch_size=4,
    eval_strategy="steps",
    num_train_epochs=3,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=3e-4,
    save_total_limit=2,
    remove_unused_columns=False  # ← 요거 추가!
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=final_dataset,
    eval_dataset=final_dataset.select(range(200)),  # 임시 평가셋 (선택)
    tokenizer=processor,
    compute_metrics=compute_metrics
)


<ipython-input-22-ba9e9ffb87de>:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
print(final_dataset.features)

{'input_values': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None), 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}


In [24]:
import wandb
os.environ["WANDB_API_KEY"] = "513a1f0c050fa7f60a76b5232e904d8df397082e"
wandb.login()

wandb: Currently logged in as: dkkim2008 (dkkim2008-hankuk-university-for-foreign-studies) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [25]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("/content/drive/MyDrive/코딩공부/project_folder/speechbrain/pretrained_model/wav2vec2-finetuned-clean100")
processor.save_pretrained("/content/drive/MyDrive/코딩공부/project_folder/speechbrain/pretrained_model/wav2vec2-finetuned-clean100")